In [1]:
import pandas as pd

comments = pd.read_csv(
    "../data/raw/kaggle_RC_2019-05.csv"
)

# Remove missing text
comments = comments.dropna(subset=["body"])

# Remove deleted/removed comments
comments = comments[
    ~comments["body"].isin(["[deleted]", "[removed]"])
]

# Remove empty/very short comments
comments["body"] = comments["body"].str.strip()

comments = comments[
    comments["body"].str.len() >= 20
]

# Remove duplicate comments
comments = comments.drop_duplicates(subset=["body"])

print("Remaining comments:", len(comments))

Remaining comments: 947872


In [2]:
comments.head()

,subreddit,body,controversiality,score
0,gameofthrones,Your submission has been automatically removed...,0,1
1,aww,"Dont squeeze her with you massive hand, you me...",0,19
2,gaming,It's pretty well known and it was a paid produ...,0,3
3,news,You know we have laws against that currently c...,0,10
4,politics,"Yes, there is a difference between gentle supp...",0,1


In [3]:
jokes = pd.read_csv(
    "../data/raw/one-million-reddit-jokes.csv"
)

# Replace missing selftext with empty string
jokes["selftext"] = jokes["selftext"].fillna("")

# Remove deleted/removed posts
jokes = jokes[
    ~jokes["selftext"].isin(["[deleted]", "[removed]"])
]

# Create the actual text we'll embed
jokes["text"] = (
    jokes["title"].fillna("") +
    "\n" +
    jokes["selftext"]
)

# Clean whitespace
jokes["text"] = jokes["text"].str.strip()

# Remove very short entries
jokes = jokes[
    jokes["text"].str.len() >= 20
]

# Remove duplicates
jokes = jokes.drop_duplicates(subset=["text"])

print("Remaining jokes:", len(jokes))

Remaining jokes: 562661


In [7]:
print(comments.columns.tolist())
print(comments.shape)

['subreddit', 'body', 'controversiality', 'score']
(947872, 4)


In [9]:
comments.groupby("subreddit").size().head()


subreddit
AmItheAsshole     23922
Animemes          23252
AskReddit         24168
ChapoTrapHouse    24300
FortNiteBR        22824
dtype: int64

In [11]:
comments_sample = (
    comments
    .groupby("subreddit", group_keys=False)
    .sample(
        n=1250,
        random_state=42
    )
    .reset_index(drop=True)
)

print("Sampled comments:", len(comments_sample))
print("Subreddits:", comments_sample["subreddit"].nunique())

Sampled comments: 50000
Subreddits: 40


In [12]:
jokes["score_bucket"] = pd.qcut(
    jokes["score"],
    q=5,
    labels=False,
    duplicates="drop"
)

jokes["score_bucket"].value_counts().sort_index()

score_bucket
0    253418
1     95599
2    103879
3    109765
Name: count, dtype: int64

In [ ]:
jokes_sample = (
    jokes
    .groupby("score_bucket", group_keys=False)
    .apply(lambda x: x.sample(
        n=min(len(x), 10000),
        random_state=42
    ))
    .reset_index(drop=True)
)

print("Sampled jokes:", len(jokes_sample))

Sampled jokes: 40000


In [15]:
print(jokes_sample.columns.tolist())

['type', 'id', 'subreddit.id', 'subreddit.name', 'subreddit.nsfw', 'created_utc', 'permalink', 'domain', 'url', 'selftext', 'title', 'score', 'text']


In [18]:
comments_sample.to_csv(
    "../data/processed/comments_clean.csv",
    index=False
)

jokes_sample.to_csv(
    "../data/processed/jokes_clean.csv",
    index=False
)

In [19]:
print("Comments:", len(comments_sample))
print("Jokes:", len(jokes_sample))

Comments: 50000
Jokes: 40000


In [20]:
print("COMMENTS")
display(
    comments_sample[["subreddit", "body", "score"]]
    .sample(10, random_state=42)
)

print("\nJOKES")
display(
    jokes_sample[["title", "selftext", "score"]]
    .sample(10, random_state=42)
)

COMMENTS


,subreddit,body,score
33553,nba,modern medicine is a miracle,1
9427,Pikabu,Да какая клиника?! ЭТО ПОЛНЫЙ ПИЗДЕЦ!!! Был по...,57
199,AmItheAsshole,"NTA- once again, but please keep us posted. I ...",8
12447,Showerthoughts,Mental illness is a software bug and viruses a...,47
39489,relationship_advice,Sounds like you suck as a human being and your...,2
42724,todayilearned,We need the information for census data. You ...,7
10822,RoastMe,Too bad the casting team for Star Wars didn’t ...,1
49498,worldnews,*The Democratic Republic of Congo has some of ...,6
4144,ChapoTrapHouse,Its funnier reading this as a woke capitalism ...,1
36958,pics,Ban guns and they make bombs. Ban bombs materi...,4



JOKES


,title,selftext,score
32823,They all laughed when I told them that one day...,If only they could see me now!,23449
16298,"The leaders of Russia, North Korea and the Uni...","Upon their arrival, they all marvel at the vie...",7
28505,Why didn’t the elephant get the job he wanted?,His qualifications were completely irrelephant.,15
6689,Hillary and Trump are in a plane crash. Who su...,America.,0
26893,The mining industry wants to put out a radio a...,"They said: ""B minor"".",11
36572,Male logic. Another joke from an 83 year old dad.,This is a conversation bet...,108
12335,Where did Kylo Ren get his lightsaber?,At the Darth Mall.\n\n^I'm ^sorry.,6
29591,What do you call a 6 year old with no friends?,A Sandy Hook survivor,12
18948,What was the chef's excuse for missing homework?,He didn't have enough thyme,5
31067,"Today, me and my wife had a .69",It would have been a hundred times better with...,2162
